In [ ]:
%pip install -q numpy matplotlib

## Scenario

This notebook covers **RNNs, LSTMs, and GRUs** -- the sequential architecture family from the Week 2 Monday deep learning material.

**Prerequisites:** You have already worked through `dl_cnn_encoder_decoder.ipynb` (Friday) which covered CNNs for image data and the encoder-decoder pattern. In that notebook, the encoder-decoder used LSTMs as a black box with a brief primer. Today you open that box.

**What this notebook covers:**
- How recurrent neural networks process sequential data with shared weights across time steps.
- The vanishing gradient problem and why vanilla RNNs struggle with long-range dependencies.
- LSTM gated memory: forget gate, input gate, cell state highway, output gate.
- GRU simplified gating: fewer parameters, comparable performance.
- A side-by-side comparison of all three architectures on a synthetic sentiment classification task.

After completing this notebook, revisit the encoder-decoder notebook to see how the LSTM internals connect to the seq2seq architecture.

## STAGE 1 -- RNNs for Sequence Data (55 min)

## STEP 1.1 -- From Images to Sequences (5 min)

## STEP 1.2 -- Generate Synthetic Sentiment Data (8 min)

In [ ]:
# Synthetic sentiment dataset
# Positive reviews: sequences with higher average token values
# Negative reviews: sequences with lower average token values

import numpy as np

VOCAB_SIZE = 50
SEQ_LEN = 20
NUM_TRAIN = 2000
NUM_TEST = 400

def generate_sentiment_data(n_samples, seq_len, vocab_size, seed=42):
    rng = np.random.RandomState(seed)
    sequences = []
    labels = []
    for _ in range(n_samples):
        label = rng.randint(0, 2)
        if label == 1:  # positive
            seq = rng.randint(vocab_size // 2, vocab_size, size=seq_len)
        else:           # negative
            seq = rng.randint(0, vocab_size // 2, size=seq_len)
        noise_idx = rng.choice(seq_len, size=seq_len // 4, replace=False)
        seq[noise_idx] = rng.randint(0, vocab_size, size=len(noise_idx))
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

X_train_seq, y_train_seq = generate_sentiment_data(NUM_TRAIN, SEQ_LEN, VOCAB_SIZE, seed=42)
X_test_seq, y_test_seq = generate_sentiment_data(NUM_TEST, SEQ_LEN, VOCAB_SIZE, seed=99)

print(f"Training sequences shape: {X_train_seq.shape}")   # (2000, 20)
print(f"Test sequences shape:     {X_test_seq.shape}")     # (400, 20)
print(f"Vocabulary size:          {VOCAB_SIZE}")
print(f"Sequence length:          {SEQ_LEN}")
print(f"Class balance:            {y_train_seq.mean():.2f} positive")
print(f"\nSample sequence:          {X_train_seq[0]}")
print(f"Sample label:             {y_train_seq[0]}")

## STEP 1.3 -- Vanilla RNN for Sentiment (12 min)

## STEP 1.4 -- The Vanishing Gradient Problem in RNNs (8 min)

In [ ]:
# Demonstrate how gradient magnitudes decay across time steps
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
seq_length = 50
W_singular_value = 0.9

gradient_norms = []
current_magnitude = 1.0
for t in range(seq_length):
    gradient_norms.append(current_magnitude)
    current_magnitude *= W_singular_value

gradient_norms = gradient_norms[::-1]

plt.figure(figsize=(10, 4))
plt.bar(range(seq_length), gradient_norms)
plt.xlabel("Time Step (earlier \u2192 later)")
plt.ylabel("Relative Gradient Magnitude")
plt.title("Gradient Flow in Vanilla RNN (50 steps, \u03c3_max = 0.9)")
plt.tight_layout()
plt.show()

print(f"Gradient at step 0 (earliest):  {gradient_norms[0]:.6f}")
print(f"Gradient at step 49 (latest):   {gradient_norms[49]:.6f}")
print(f"Ratio (earliest/latest):        {gradient_norms[0] / gradient_norms[49]:.6f}")

## STEP 1.5 -- LSTM: Gated Memory (12 min)

## STEP 1.6 -- GRU: Simplified Gating (5 min)

## STEP 1.7 -- Compare All Three Architectures (5 min)

In [ ]:
# Visualization of expected comparison (conceptual)
import matplotlib.pyplot as plt

epochs = list(range(1, 21))
# Simulated representative curves
losses_rnn  = [0.7 - 0.015*e + 0.002*e**0.5 for e in epochs]
losses_lstm = [0.7 - 0.025*e + 0.003*e**0.5 for e in epochs]
losses_gru  = [0.7 - 0.027*e + 0.003*e**0.5 for e in epochs]

accs_rnn  = [0.5 + 0.02*e - 0.0005*e**2 for e in epochs]
accs_lstm = [0.5 + 0.025*e - 0.0005*e**2 for e in epochs]
accs_gru  = [0.5 + 0.024*e - 0.0005*e**2 for e in epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, losses_rnn, label="Vanilla RNN")
ax1.plot(epochs, losses_lstm, label="LSTM")
ax1.plot(epochs, losses_gru, label="GRU")
ax1.set_title("Training Loss (Representative)"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, accs_rnn, label="Vanilla RNN")
ax2.plot(epochs, accs_lstm, label="LSTM")
ax2.plot(epochs, accs_gru, label="GRU")
ax2.set_title("Test Accuracy (Representative)"); ax2.set_xlabel("Epoch"); ax2.legend()

plt.tight_layout(); plt.show()